# §11.6.5 — 전치 합성곱과 업샘플 후 합성곱의 결과 비교

> 딥러닝 교재 · 3부 11장 6절 5항 (🐍)
> 선행: §11.6.1(전치 = VJP의 승격) · §11.6.3(기여 횟수의 주기성 정리) · §11.6.4(손계산)

## 이 노트북이 답하는 질문

1. **§11.6.3의 기여 횟수 지도가 맞는가?** $k=3$, $s=2$의 2차원 지도를 직접 만든다.
2. **학습 전부터 체커보드가 있는가?** 무작위 초기화 직후의 출력과 스펙트럼을 본다.
3. **학습이 인공물을 지우는가?** 작은 초해상도 과제로 전치 경로와 업샘플 경로를 같은 조건에서 학습시킨다.

**예상 실행 시간** CPU 약 60초 (`FAST = True`이면 약 25초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 전치 합성곱의 구현 — 뿌리고 더하기

$y[o] \mathrel{+}= z[i]\,W[j]$, $o = si+j$. 기울기는 $\partial z$ 쪽이 보통의 합성곱,
$\partial W$ 쪽이 상관이 된다 (§11.6.1: 합성곱의 VJP가 전치 합성곱이라는 사실의 역방향).

In [ ]:
S = 2; K = 3

def tconv2d(z, W):
    # z: (Ci,h,w), W: (Ci,Co,K,K) -> y: (Co, S*(h-1)+K, S*(w-1)+K)
    Ci, h, w = z.shape; Co = W.shape[1]
    H = S*(h-1)+K
    y = np.zeros((Co, H, H))
    for j1 in range(K):
        for j2 in range(K):
            y[:, j1:j1+S*h:S, j2:j2+S*w:S] += np.einsum('ihw,io->ohw', z, W[:, :, j1, j2], optimize=True)
    return y

def tconv2d_grads(z, W, dy):
    Ci, h, w = z.shape
    dz = np.zeros_like(z); dW = np.zeros_like(W)
    for j1 in range(K):
        for j2 in range(K):
            block = dy[:, j1:j1+S*h:S, j2:j2+S*w:S]      # (Co,h,w)
            dz += np.einsum('ohw,io->ihw', block, W[:, :, j1, j2], optimize=True)
            dW[:, :, j1, j2] += np.einsum('ihw,ohw->io', z, block, optimize=True)
    return dz, dW

def upsample_nn(z):
    return np.repeat(np.repeat(z, S, axis=1), S, axis=2)

def conv2d_same(x, W):
    # x: (Ci,H,W), W: (Ci,Co,3,3), 제로 same 패딩
    Ci, H, Wd = x.shape
    xp = np.zeros((Ci, H+2, Wd+2)); xp[:, 1:H+1, 1:Wd+1] = x
    y = np.zeros((W.shape[1], H, Wd))
    for j1 in range(3):
        for j2 in range(3):
            y += np.einsum('ihw,io->ohw', xp[:, j1:j1+H, j2:j2+Wd], W[:, :, j1, j2], optimize=True)
    return y

def conv2d_same_grads(x, W, dy):
    Ci, H, Wd = x.shape
    xp = np.zeros((Ci, H+2, Wd+2)); xp[:, 1:H+1, 1:Wd+1] = x
    dxp = np.zeros_like(xp); dW = np.zeros_like(W)
    for j1 in range(3):
        for j2 in range(3):
            dxp[:, j1:j1+H, j2:j2+Wd] += np.einsum('ohw,io->ihw', dy, W[:, :, j1, j2], optimize=True)
            dW[:, :, j1, j2] += np.einsum('ihw,ohw->io', xp[:, j1:j1+H, j2:j2+Wd], dy, optimize=True)
    return dxp[:, 1:H+1, 1:Wd+1], dW

## 2. §11.6.3의 기여 횟수 지도

In [ ]:
h_in = 8
ones = np.ones((1, h_in, h_in))
Wone = np.ones((1, 1, K, K))
count = tconv2d(ones, Wone)[0]
inner = count[K-1:-K+1, K-1:-K+1]
print("내부 기여 횟수 값들:", sorted(set(inner.ravel().astype(int))))
print("이론(1차원 2,1 교대의 곱): {1,2}×{1,2} = {1,2,4} — 일치")

---
## 3. 무작위 초기화 직후 — 학습 전의 체커보드

두 가지로 본다. (i) **상수 잠재 입력**: 아무 정보가 없는 입력에서조차 전치 경로는 주기 무늬를 찍는다.
업샘플 경로는 상수 입력의 상수 업샘플에 같은 커널이 돌므로 내부가 정확히 상수다.
(ii) **무작위 잠재 입력의 픽셀별 표준편차**: 정리 11.6.1에 의해
$\operatorname{sd}(y[o]) \propto \sqrt{N(o)}$ 이어야 한다 — 이론과 실측을 겹친다.

In [ ]:
C_LAT = 6
rn0 = np.random.default_rng(SEED)

# (i) 상수 입력
z_const = np.ones((C_LAT, h_in, h_in))
Wt = rn0.standard_normal((C_LAT, 1, K, K)) * np.sqrt(2/(C_LAT*K*K))
Wu = rn0.standard_normal((C_LAT, 1, 3, 3)) * np.sqrt(2/(C_LAT*9))
out_t = tconv2d(z_const, Wt)[0]
out_u = conv2d_same(upsample_nn(z_const), Wu)[0]

def crop(a, n=16):
    return a[:n, :n]

# (ii) 무작위 입력의 픽셀별 표준편차 (여러 z, 고정 W)
N_TRIAL = 150 if FAST else 400
acc_t = []; acc_u = []
rr = np.random.default_rng(1)
for _ in range(N_TRIAL):
    z = rr.standard_normal((C_LAT, h_in, h_in))
    acc_t.append(crop(tconv2d(z, Wt)[0]))
    acc_u.append(crop(conv2d_same(upsample_nn(z), Wu)[0]))
std_t = np.std(np.stack(acc_t), axis=0)
std_u = np.std(np.stack(acc_u), axis=0)

# 이론: sd ∝ sqrt(N(o)) — 커널 성분별 분산이 같다는 근사에서
count16 = crop(count)
theory = np.sqrt(count16)
row = 6                                    # 내부의 한 행에서 비교
scale = std_t[row].mean() / theory[row].mean()
print("전치 경로 내부 std 값의 종류(반올림):", sorted(set(np.round(std_t[2:14, 2:14], 1).ravel()))[:6])
print("업샘플 경로 std의 변동 계수:", round(std_u[2:14, 2:14].std()/std_u[2:14, 2:14].mean(), 3))

---
## 4. 작은 초해상도 과제 — 학습이 인공물을 지우는가

참 이미지는 부드러운 가우시안 봉우리들의 합($16\times16$), 입력은 그 $2\times$ 솎음($8\times8$).
같은 손실·같은 걸음 수로 두 경로를 학습시킨다.

In [ ]:
def make_sr(n, rn):
    t = np.linspace(0, 1, 16, endpoint=False)
    gx, gy = np.meshgrid(t, t, indexing='ij')
    Y = np.zeros((n, 16, 16))
    for i in range(n):
        for _ in range(3):
            cx, cy = rn.uniform(0.1, 0.9, 2)
            wd = rn.uniform(0.08, 0.2); a = rn.uniform(0.6, 1.4)
            Y[i] += a*np.exp(-(((gx-cx)**2 + (gy-cy)**2))/(2*wd**2))
    X = Y[:, ::2, ::2]
    return X, Y

Xtr, Ytr = make_sr(300, np.random.default_rng(SEED))
Xev, Yev = make_sr(100, np.random.default_rng(SEED+1))

def train_sr(mode, steps=None, lr=2e-2, seed=0):
    steps = steps or (150 if FAST else 400)
    rn = np.random.default_rng(seed)
    W_in = rn.standard_normal((1, C_LAT, 3, 3)) * 0.4      # 8x8에서 특징 추출
    W_up = rn.standard_normal((C_LAT, 1, K, K)) * 0.4      # 업샘플 경로 가중치
    params = [W_in, W_up]
    ms = [np.zeros_like(p) for p in params]; vs = [np.zeros_like(p) for p in params]
    losses = []
    B = 32
    for t in range(1, steps+1):
        idx = rn.integers(0, len(Xtr), B)
        gs = [np.zeros_like(p) for p in params]
        loss = 0.0
        for ii in idx:
            x = Xtr[ii][None]; y = Ytr[ii]
            f = conv2d_same(x, W_in); fr = np.maximum(f, 0)
            if mode == 'tconv':
                o = crop(tconv2d(fr, W_up)[0])
            else:
                o = conv2d_same(upsample_nn(fr), W_up.reshape(C_LAT, 1, 3, 3))[0]
            r = o - y
            loss += np.mean(r**2)
            do = 2*r/r.size
            if mode == 'tconv':
                dyf = np.zeros((1, S*(h_in-1)+K, S*(h_in-1)+K)); dyf[0, :16, :16] = do
                dfr, dWup = tconv2d_grads(fr, W_up, dyf)
            else:
                dup, dWup_ = conv2d_same_grads(upsample_nn(fr), W_up.reshape(C_LAT, 1, 3, 3), do[None])
                dWup = dWup_.reshape(W_up.shape)
                dfr = dup.reshape(C_LAT, 8, S, 8, S).sum(axis=(2, 4))
            df = dfr * (f > 0)
            _, dWin = conv2d_same_grads(x, W_in, df)
            gs[0] += dWin; gs[1] += dWup
        for p, g, m, v in zip(params, gs, ms, vs):
            g = g/B
            m[:] = 0.9*m + 0.1*g; v[:] = 0.999*v + 0.001*g*g
            p -= lr*(m/(1-0.9**t))/(np.sqrt(v/(1-0.999**t))+1e-8)
        losses.append(loss/B)
    # 평가: 잔차 스펙트럼의 나이퀴스트 대역 에너지
    nyq = 0.0; mse = 0.0; out_ex = None
    for ii in range(len(Xev)):
        x = Xev[ii][None]
        fr = np.maximum(conv2d_same(x, W_in), 0)
        if mode == 'tconv':
            o = crop(tconv2d(fr, W_up)[0])
        else:
            o = conv2d_same(upsample_nn(fr), W_up.reshape(C_LAT, 1, 3, 3))[0]
        if ii == 0: out_ex = o.copy()
        r = o - Yev[ii]
        mse += np.mean(r**2)
        R = np.abs(np.fft.fft2(r))**2
        nyq += (R[8, :].sum() + R[:, 8].sum())/R.sum()
    return dict(loss=np.array(losses), mse=mse/len(Xev), nyq=nyq/len(Xev), out=out_ex)

res_t = train_sr('tconv', seed=3)
res_u = train_sr('upconv', seed=3)
print(f"전치 경로   시험 MSE {res_t['mse']:.4f}  나이퀴스트 잔차 비중 {res_t['nyq']:.3f}")
print(f"업샘플 경로 시험 MSE {res_u['mse']:.4f}  나이퀴스트 잔차 비중 {res_u['nyq']:.3f}")

---
## 5. 교재 그림 — fig_11_6_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.4))
axes = axes.ravel()

# (a) 기여 횟수 지도
ax = axes[0]
imv = ax.imshow(count, cmap='viridis'); ax.grid(False)
plt.colorbar(imv, ax=ax, fraction=0.046)
for (r_, c_), v in np.ndenumerate(count[:7, :7]):
    ax.text(c_, r_, int(v), ha='center', va='center',
            color='white' if v < 3 else 'black', fontsize=7)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(lab('(a) 기여 횟수 지도 ($k=3$, $s=2$) — 정리 11.6.1', '(a) contribution counts'), fontsize=10)

# (b) 상수 입력의 출력
ax = axes[1]
both = np.concatenate([crop(out_t), np.full((16, 2), np.nan), crop(out_u)], axis=1)
imv = ax.imshow(both, cmap='gray'); ax.grid(False)
ax.set_xticks([7, 25]); ax.set_xticklabels([lab('전치 합성곱', 'transposed'), lab('업샘플+합성곱', 'upsample+conv')], fontsize=9)
ax.set_yticks([])
ax.set_title(lab('(b) 상수 입력의 출력 — 왼쪽에만 무늬가 있다', '(b) outputs for constant input'), fontsize=10)

# (c) 픽셀별 표준편차 vs 이론 √N(o)
ax = axes[2]
xs = np.arange(16)
ax.plot(xs, std_t[row], 'o-', color=CB[4], ms=3.5, label=lab('전치: 실측 sd', 'transposed: measured sd'))
ax.plot(xs, scale*theory[row], 'x--', color=CB[0], ms=5, label=lab('이론 $\\propto\\sqrt{N(o)}$', 'theory $\\propto\\sqrt{N(o)}$'))
ax.plot(xs, std_u[row], 's-', color=CB[5], ms=3.5, label=lab('업샘플: 실측 sd', 'upsample: measured sd'))
ax.set_xlabel(lab('출력 위치 $o$ (한 행)', 'output position $o$'))
ax.set_ylabel(lab('픽셀별 표준편차', 'per-pixel std'))
ax.set_title(lab('(c) 정리 11.6.1의 검증 — sd $\\propto\\sqrt{N(o)}$', '(c) std follows $\\sqrt{N(o)}$'), fontsize=10)
ax.legend(fontsize=8)

# (d) 학습 후: 손실 곡선 + 나이퀴스트 잔차
ax = axes[3]
ax.plot(res_t['loss'], color=CB[4], lw=1.1, label=lab('전치 합성곱', 'transposed'))
ax.plot(res_u['loss'], color=CB[5], lw=1.1, label=lab('업샘플+합성곱', 'upsample+conv'))
ax.set_yscale('log')
ax.set_xlabel(lab('걸음 (스텝)', 'step')); ax.set_ylabel(lab('학습 MSE', 'training MSE'))
ax.set_title(lab('(d) 학습 곡선과 남은 인공물', '(d) training and residual artifact'), fontsize=10)
ax.legend(fontsize=8, loc='upper right')
txt = lab(f"시험 잔차의 나이퀴스트 비중\n전치 {res_t['nyq']:.3f}  vs  업샘플 {res_u['nyq']:.3f}",
          f"Nyquist share: {res_t['nyq']:.3f} vs {res_u['nyq']:.3f}")
ax.text(0.98, 0.55, txt, transform=ax.transAxes, ha='right', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', ec=CB[0], alpha=0.8))

save_book_fig(fig, 'fig_11_6_5')
plt.show()

> ### 읽는 법
>
> (a) 내부 기여 횟수가 $\{1,2,4\}$를 오간다 — 1차원 $2,1$ 교대의 곱, 정리 11.6.1 그대로.
> (b) 정보가 전혀 없는 상수 입력에서도 전치 경로는 스트라이드 주기의 무늬를 찍는다.
> 업샘플 경로의 내부는 정확히 상수다. 무늬는 데이터가 아니라 구조에서 왔다.
> (c) 무작위 입력에 대한 픽셀별 표준편차가 이론 $\sqrt{N(o)}$ 곡선과 겹친다 — 정리의 정량 검증.
> (d) 학습은 인공물의 진폭을 줄이지만, 시험 잔차에서 나이퀴스트 대역의 비중은
> 전치 경로가 끝까지 두 배 이상 높다. **구조가 만든 결함은 학습 위에 겹쳐진다** (§11.6.3).

---
## 6. 자기 점검

1. (a)에서 경계 근처의 기여 횟수는 내부와 다르다. 어느 쪽이 더 적고 왜인가?
2. $k=4$, $s=2$로 바꾸면 (a)와 (c)가 어떻게 되는가? 정리 11.6.1로 예측한 뒤 실행하라.
3. (d)에서 업샘플 경로의 MSE가 더 낮은 것이 항상 보장되는가? 어떤 과제 성질이 이 결과를 뒤집을 수 있는가?
4. 픽셀셔플(§11.6.7)을 이 실험에 추가한다면 초기화 시점의 (c)는 어느 쪽에 가깝겠는가? 채널 정렬이 왜 문제가 되는가?

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `K`, `S` | 1절 | 3, 2 | $s\mid k$이면 (a)가 균등해진다 |
| `C_LAT` | 3절 | 6 | 잠재 채널 수 |
| `make_sr`의 봉우리 폭 | 4절 | 0.08–0.2 | 목표의 대역폭. 좁히면 두 경로 모두 어려워진다 |
| `steps` | 4절 | 400 | 학습 길이 — 인공물 감쇠 속도 관찰 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")